In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from bayesian_torch.layers.flipout_layers import conv_flipout as bnn_conv, linear_flipout as bnn_linear

class LinearMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()

        self.in_features = original_layer.in_features
        self.out_features = original_layer.out_features

        self.w_mu = nn.Parameter(original_layer.mu_weight.clone()) 
        self.w_var = nn.Parameter(F.softplus(original_layer.rho_weight).pow(2))

        self.b_mu = nn.Parameter(original_layer.mu_bias.clone())
        self.b_var = nn.Parameter(F.softplus(original_layer.rho_bias).pow(2))

    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu = F.linear(x_mu, self.w_mu, self.b_mu)

        r_var = F.linear(x_var, self.w_var, self.b_var) 
        r_var += F.linear(x_var, self.w_mu.pow(2)) 
        r_var += F.linear(x_mu.pow(2), self.w_var)

        return torch.stack((r_mu, r_var))


class Conv2dMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()

        self.groups = original_layer.groups
        self.padding = original_layer.padding
        self.stride = original_layer.stride

        self.w_mu = nn.Parameter(original_layer.mu_kernel.clone())
        self.w_var = nn.Parameter(F.softplus(original_layer.rho_kernel).pow(2))

        if original_layer.mu_bias != None:
            self.b_mu = nn.Parameter(original_layer.mu_bias.clone())
            self.b_var = nn.Parameter(F.softplus(original_layer.rho_bias).pow(2))
        else:
            self.b_mu = None
            self.b_var = None

    def batch_norm_fold(self, bn_layer):
        bn_coef = bn_layer.weight / torch.sqrt(bn_layer.running_var + bn_layer.eps)
        bn_coef_k = bn_coef.view(-1,1,1,1).expand(self.w_mu.shape)

        self.w_mu = nn.Parameter(self.w_mu * bn_coef_k)
        self.w_var = nn.Parameter((torch.sqrt(self.w_var) * bn_coef_k).pow(2))
    
        if self.b_mu != None:
            self.b_mu = nn.Parameter((self.b_mu - bn_layer.running_mean) * bn_coef + bn_layer.bias)
            self.b_var = nn.Parameter((torch.sqrt(self.b_var) * bn_coef).pow(2))
        else:
            self.b_mu = nn.Parameter((0 - bn_layer.running_mean) * bn_coef + bn_layer.bias)
            self.b_var = nn.Parameter((0 * bn_coef).pow(2))

    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu = F.conv2d(x_mu, self.w_mu, self.b_mu, stride=self.stride, padding=self.padding, groups=self.groups)
       
        r_var = F.conv2d(x_var, self.w_var, self.b_var, stride=self.stride, padding=self.padding, groups=self.groups) 
        r_var += F.conv2d(x_var, self.w_mu.pow(2), stride=self.stride, padding=self.padding, groups=self.groups)
        r_var += F.conv2d(x_mu.pow(2), self.w_var, stride=self.stride, padding=self.padding, groups=self.groups)

        return torch.stack((r_mu, r_var))

class BatchNorm2dMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()
        self.gamma = nn.Parameter(original_layer.weight)
        self.beta = nn.Parameter(original_layer.bias)
        self.running_mean = nn.Buffer(original_layer.running_mean)
        self.running_var = nn.Buffer(original_layer.running_var)
        self.eps = original_layer.eps
        self.momentum = original_layer.momentum

    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        # Defined p(z) = mean(p(x)) across batch dimension
        
        if self.training:
            batch_size = x_mu.shape[0]
            z_mu = x_mu.mean(dim=0)
            z_var = (x_mu.pow(2) + x_var).mean(dim=0) - x_mu.sum(0).pow(2) / (batch_size**2)
            
            # Update running mean and running var using momentum
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * z_mu
            self.running_var = (1 - self.momentum) * self.running_var + self.running_var * z_var
        else:
            z_mu = self.running_mean
            z_var = self.running_var

        r_mu = self.gamma * (x_mu - z_mu) / torch.sqrt(z_var + self.eps) + self.beta
        r_var = self.gamma.pow(2) * x_var / (z_var + self.eps)

        return torch.stack((r_mu, r_var))


class AvgPool2dMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()
        self.kernel_size = original_layer.kernel_size
        self.stride = original_layer.stride
    
    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu = F.avg_pool2d(x_mu, self.kernel_size, self.stride)
        r_var = F.avg_pool2d(x_var, self.kernel_size, self.stride) / (self.kernel_size**2)

        return torch.stack((r_mu, r_var))
    

class AdaptiveAvgPool2dMomentProp(nn.Module):
    def __init__(self, original_layer):
        super().__init__()
        self.output_size = original_layer.output_size
    
    def forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        _, _, h, w = x_mu.shape
        ishape = torch.tensor([h, w])

        stride = torch.floor(ishape / self.output_size)
        kernel_size = int((ishape - (self.output_size - 1) * stride).prod())

        r_mu = F.adaptive_avg_pool2d(x_mu, self.output_size)
        r_var = F.adaptive_avg_pool2d(x_var, self.output_size) / (kernel_size**2)

        return torch.stack((r_mu, r_var))

def _online_mean_var(sample_f, nsamples):
    # Online shifted variance calculation (r, r2, k)
    k = sample_f()
    r = torch.zeros_like(k)
    r2 = torch.zeros_like(k)
    
    for _ in range(nsamples - 1):
        x_sample = sample_f()
        r += x_sample - k
        r2 += (x_sample - k).pow(2)

    r_mu = k + r / nsamples
    r_var = (r2 - r.pow(2) / nsamples) / (nsamples - 1)

    return torch.stack((r_mu, r_var))


class MaxPoolMomentProp(nn.Module):
    def __init__(self, original_layer, num_mc=100, use_simple=False, use_large=False):
        super().__init__()
        self.num_mc = num_mc
        self.use_simple = use_simple
        self.use_large = use_large
        self.kernel_size = original_layer.kernel_size
        self.stride = original_layer.stride
    
    def mc_forward(self, x):
        x_mu = x[0,:]
        x_std = torch.sqrt(x[1,:])

        def _large_f():
            s = torch.empty((self.num_mc, *x_mu[0,:].shape), device=x_mu.device)
            s = s.normal_().unsqueeze(1).expand(self.num_mc, *x_mu.shape)
            s = torch.addcmul(x_mu.unsqueeze(0).expand(self.num_mc, *x_mu.shape), s, x_std.unsqueeze(0).expand(self.num_mc, *x_std.shape))
            
            # Flat the samples and the batch in the same first dimension
            s = s.view(self.num_mc * s.shape[1], *s[0,0,:].shape)
            s = F.max_pool2d(s, self.kernel_size, self.stride)
            # Divide the first 2 dimensions again in samples, batch
            s = s.view(self.num_mc, x_mu.shape[0], *s[0].shape)

            return torch.stack((s.mean(dim=0), s.var(dim=0)))

        # Allocate tensor in device and share samples across batch
        def _batch_f():
            n = torch.empty_like(x_mu[0,:]).normal_().unsqueeze(0).expand(*x_mu.shape)
            s = n * x_std + x_mu
            return F.max_pool2d(s, self.kernel_size, self.stride)
    
        if self.use_large:
            return _large_f()
        else:
            return _online_mean_var(_batch_f, self.num_mc)
    
    def simple_forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu, idx = F.max_pool2d(x_mu, self.kernel_size, self.stride, return_indices=True)
        mask = F.max_unpool2d(torch.ones_like(r_mu), idx, self.kernel_size, self.stride)
        r_var = F.max_pool2d(mask * x_var, self.kernel_size, self.stride)

        return torch.stack((r_mu, r_var))

    def forward(self, x):
        if self.use_simple:
            return self.simple_forward(x)
        else:
            return self.mc_forward(x)


class ReLUMomentProp(nn.Module):
    def __init__(self, num_mc=100, use_simple=False, use_large=False):
        super().__init__()
        self.num_mc = num_mc
        self.use_simple = use_simple
        self.use_large = use_large

    def mc_forward(self, x):
        x_mu = x[0,:]
        x_std = torch.sqrt(x[1,:])

        def _large_f():
            r = torch.empty((self.num_mc, *x_mu[0,:].shape), device=x_mu.device)
            r = r.normal_().unsqueeze(1).expand(self.num_mc, *x_mu.shape)
            r = torch.addcmul(x_mu.unsqueeze(0).expand(self.num_mc, *x_mu.shape), r, x_std.unsqueeze(0).expand(self.num_mc, *x_std.shape))
            r = F.relu(r)
            return torch.stack((r.mean(dim=0), r.var(dim=0)))

        def _batch_f():
            n = torch.empty_like(x_mu[0,:]).normal_().unsqueeze(0).expand(*x_mu.shape)
            s = n * x_std + x_mu
            return F.relu(s)
        
        if self.use_large:
            return _large_f()
        else:
            return _online_mean_var(_batch_f, self.num_mc)
    
    def simple_forward(self, x):
        x_mu = x[0,:]
        x_var = x[1,:]

        r_mu = F.relu(x_mu)
        r_var = x_var * (x_mu > 0)

        return torch.stack((r_mu, r_var))

    def forward(self, x):
        if self.use_simple:
            return self.simple_forward(x)
        else:
            return self.mc_forward(x)


class SoftmaxMomentProp(nn.Module):
    def __init__(self, num_mc=100):
        super().__init__()
        self.num_mc = num_mc

    def forward(self, x):
        x_mu = x[0,:]
        x_std = torch.sqrt(x[1,:])

        r = torch.empty((self.num_mc, *x_mu[0,:].shape), device=x_mu.device)
        r = r.normal_().unsqueeze(1).expand(self.num_mc, *x_mu.shape)
        r = torch.addcmul(x_mu.unsqueeze(0).expand(self.num_mc, *x_mu.shape), r, x_std.unsqueeze(0).expand(self.num_mc, *x_std.shape))
        return F.softmax(r, dim=2)


class BnnSampler(nn.Module):
    def __init__(self, layer, num_mc=100, return_mp=False, use_large=False):
        super().__init__()
        self.num_mc = num_mc 
        self.l = layer
        self.use_large = use_large
        self.return_mp = return_mp

    def forward(self, x):
        x_mu = x[0,:]
        x_std = torch.sqrt(x[1,:])

        r = []
        for _ in range(self.num_mc):
            s = torch.addcmul(x_mu, torch.empty_like(x_mu).normal_(), x_std)
            r.append(self.l(s))
        r = torch.stack(r)
        
        if self.return_mp:
            return torch.stack((r.mean(dim=0), r.var(dim=0)))
        else:
            return r

def bnn_to_mp(m, num_mc, use_simple=False, use_large=False):
    prev = None
    for name, t in list(m._modules.items()):
        # Recursive call (modules inside module)
        if m._modules[name]._modules:
            if isinstance(t, BnnSampler):
                pass
            else:
                bnn_to_mp(m._modules[name], num_mc=num_mc, use_simple=use_simple, use_large=use_large)

        if isinstance(t, bnn_linear.LinearFlipout):
            setattr(m, name, LinearMomentProp(t))
        elif isinstance(t, bnn_conv.Conv2dFlipout):
            setattr(m, name, Conv2dMomentProp(t))
        elif isinstance(t, nn.AvgPool2d):
            setattr(m, name, AvgPool2dMomentProp(t))
        elif isinstance(t, nn.AdaptiveAvgPool2d):
            setattr(m, name, AdaptiveAvgPool2dMomentProp(t))
        
        elif isinstance(t, nn.Sigmoid):
            setattr(m, name, BnnSampler(t, num_mc=num_mc, return_mp=True))
        elif isinstance(t, nn.SiLU):
            setattr(m, name, BnnSampler(t, num_mc=num_mc, return_mp=True))

        elif isinstance(t, nn.ReLU):
            l = ReLUMomentProp()
            l.use_large = use_large
            l.use_simple = use_simple
            l.num_mc = num_mc
            setattr(m, name, l)
        elif isinstance(t, nn.MaxPool2d):
            l =  MaxPoolMomentProp(t)
            l.use_large = use_large
            l.use_simple = use_simple
            l.num_mc = num_mc
            setattr(m, name, l)
        elif isinstance(t, nn.Softmax):
            l = SoftmaxMomentProp()
            l.num_mc = num_mc
            setattr(m, name, l)

        elif isinstance(t, nn.BatchNorm2d):
            prev_m, prev_name = prev
            prev_m._modules[prev_name].batch_norm_fold(t)
            setattr(m, name, nn.Identity())

        prev = (m, name)

def split_model_graph(model: nn.Module, n: int):
    """
    AI generated function.
    Splits a PyTorch model into two independently runnable models.
    Part B will contain the last N executed module calls.
    Part A will contain everything before that.
    """
    traced = torch.fx.symbolic_trace(model)
    nodes = list(traced.graph.nodes)
    
    # Find all actual module calls to determine the split point
    module_nodes = [node for node in nodes if node.op == 'call_module']
    if n >= len(module_nodes):
        raise ValueError(f"Cannot split off {n} layers; model only has {len(module_nodes)} module calls.")
        
    # The first node of Part B is the n-th module from the end
    first_node_of_b = module_nodes[-n]
    cut_idx = nodes.index(first_node_of_b)
    
    nodes_A = nodes[:cut_idx]
    nodes_B = nodes[cut_idx:]
    
    # 1. Identify "Boundary Nodes" 
    # (Tensors calculated in A that are needed in B)
    boundary_nodes = []
    for node in nodes_B:
        for in_node in node.all_input_nodes:
            if in_node in nodes_A and in_node not in boundary_nodes:
                boundary_nodes.append(in_node)
                
    # 2. Build Part A
    graph_A = torch.fx.Graph()
    env_A = {} # Maps old graph nodes to new Graph A nodes
    
    for node in nodes_A:
        env_A[node] = graph_A.node_copy(node, lambda x: env_A[x])
        
    # Set the outputs for Part A based on the boundary nodes we found
    output_args_A = tuple(env_A[node] for node in boundary_nodes)
    if len(output_args_A) == 1:
        graph_A.output(output_args_A[0])
    else:
        graph_A.output(output_args_A)
        
    model_A = torch.fx.GraphModule(traced, graph_A)
    
    # 3. Build Part B
    graph_B = torch.fx.Graph()
    env_B = {} # Maps old graph nodes to new Graph B nodes
    
    # Create input placeholders in B for the incoming tensors from A
    for node in boundary_nodes:
        env_B[node] = graph_B.placeholder(node.name)
        
    # Copy the remaining nodes into B
    for node in nodes_B:
        env_B[node] = graph_B.node_copy(node, lambda x: env_B[x])
        
    model_B = torch.fx.GraphModule(traced, graph_B)
    
    return model_A, model_B

def copy_model_params_by_name(src: nn.Module, dst: nn.Module):
    with torch.no_grad():
        
        dst_dict = dict(dst.named_parameters())
        src_dict = dict(src.named_parameters())
        
        for k in dst_dict:
            dst.get_parameter(k).copy_(src_dict[k])

        dst_dict = dict(dst.named_buffers())
        src_dict = dict(src.named_buffers())

        for k in dst_dict:
            dst.get_buffer(k).copy_(src_dict[k])


from torchvision.models import efficientnet_b0
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn

# BNN conversion hyperparmeters
bnn_prior_parameters = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Flipout",  # Flipout or Reparameterization
    "moped_enable": False,  # True to initialize mu/sigma from the pretrained dnn weights
    "moped_delta": 0.5,
}

class MomentPropEfficientnet_b0(nn.Module):
    def __init__(self, original, num_mc=20):
        super().__init__()
        self.num_mc = num_mc

        new_model = efficientnet_b0()
        new_model.classifier[1] = nn.Linear(1280, 9)

        new_model.eval()
        head, tail = split_model_graph(new_model, 3)
        
        dnn_to_bnn(head, bnn_prior_parameters)
        self.head = head
        copy_model_params_by_name(original, self.head)
        bnn_to_mp(self.head, num_mc, False, False)

        dnn_to_bnn(tail, bnn_prior_parameters)
        copy_model_params_by_name(original, tail)
        self.tail = BnnSampler(tail, num_mc=num_mc)

    def forward(self, x: torch.Tensor):
        y = torch.stack((x, torch.zeros_like(x)))
        y = self.head(y)
        return self.tail(y)

In [19]:
import torch
import torchvision
import numpy as np
import torch.nn as nn
import time
import os
import json
import subprocess
import threading
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
import matplotlib.pyplot as plt
import io
from openpyxl.drawing.image import Image as XLImage
from torchvision.transforms import v2
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn
from medmnist import PathMNIST, INFO


In [20]:
lr = 0.005
epochs = 50
batch_size = 128
milestones = [20, 35, 45]
gamma = 0.5
num_monte_carlo = 100
nb_couches = 32

In [21]:
batch_size = 128
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
info       = INFO['pathmnist']
testset    = PathMNIST(split="test", download=True, size=28, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                          shuffle=False, num_workers=4, pin_memory=True)
#device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

In [22]:
device = torch.device("cpu")

In [23]:
net = torchvision.models.efficientnet_b0(progress=True)
net.classifier[1] = nn.Linear(1280, 9)

const_bnn_prior_parameters = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": True,
    "moped_delta": 0.5,
}

dnn_to_bnn(net, const_bnn_prior_parameters)

red_bay = torch.load("BayMoped_full_run0.pth", map_location='cpu')

net.load_state_dict(red_bay['model_state_dict'])



<All keys matched successfully>

In [24]:
net = MomentPropEfficientnet_b0(net, num_mc=100)

In [ ]:
def evaluate(net, device):
    net.eval()
    all_preds = []
    all_labels = []
    all_outputs = []

    with torch.no_grad():
        for i, data in enumerate(testloader, 0):
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1)
                        
            logits = net(inputs)
            output = torch.nn.functional.softmax(logits, dim=-1) 
            pred_mean = output.mean(dim=0)                       
            y_pred = torch.argmax(pred_mean, dim=1)               

            all_preds.append(y_pred.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_outputs.append(output.cpu().numpy())         

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_outputs = np.concatenate(all_outputs, axis=1)

    acc = (all_preds == all_labels).mean()

    return all_preds, all_labels, all_outputs, acc

all_preds, all_labels, all_outputs, test_acc = evaluate(net, device)

print(f'Test Accuracy  : {100 * test_acc:.3f} %')

Test Accuracy  : 72.702 %
